# SafeCrop · Step 4: every level on its own, three pipelines, and different angles

Models: the **CLIP photo database** (free, inside Colab) and **Gemini 3.5 Flash-Lite** (free AI Studio key, no card).

Last week the loop always started at L3 and moved to a clearer photo only when the model wasn't confident. This week's version follows the supervisor's three changes:

**1. Every level on its own (no level bias).** Each photo is tested at L3, L2, L1 and L0 separately. Nothing is carried over from another level, so each level gets a clean accuracy of its own.

**2. Three pipelines.** Each pipeline starts at a different level and moves to a clearer photo only while the model isn't confident:

| Pipeline | Starts at | Then, if not confident |
|---|---|---|
| **P3** | L3 (most damaged) | L2 → L1 → L0 |
| **P2** | L2 | L1 → L0 |
| **P1** | L1 | L0 (clean) |

When a pipeline moves to the next level, it brings along the **features from the level it just left**:
* **CLIP:** the previous photo's feature vector (the numbers CLIP uses to describe a picture) is mixed 50/50 with the new one.
* **Gemini:** its own notes from the previous photo (what it saw, what it guessed and how sure it was) are given to it with the new photo.

**3. Different angles.** Every test above is repeated with the same photo turned a quarter, turned upside down and mirrored (like seeing the leaf from the other side). The photos aren't new: they are the same photo, rotated or flipped.

Comparing the three pipelines with the levels on their own shows how much carrying features helps or biases the answer at each level, and comparing angles shows whether the models depend on how the photo is held.

### Before you run
1. **The images:** the same `noisy_images.zip` as before, anywhere in your Google Drive.
2. **CLIP** runs inside Colab and is free. A GPU makes it faster: **Runtime → Change runtime type → T4 GPU**.
3. **Gemini, with a free key (no card):** go to **aistudio.google.com → Get API key → Create API key** and copy it. In Colab, click the **key icon** on the left → **Add new secret**, name it `GEMINI_API_KEY`, paste the key and switch on **Notebook access**.
4. Click **Runtime → Run all**. Colab asks you to sign in for Drive.

**How long:**
* CLIP: about 5 minutes on a GPU, all four angles.
* Gemini 3.5 Flash-Lite (original angle): about 1,450 questions in total. The free key allows **15 a minute and 500 a day**, so each day's 500 take about 40 minutes, and the whole run takes about 3 days.
* Gemini asks in two parts. **Part 1** tests every image at every level on its own (840 questions: "test all the images"). **Part 2** runs the three pipelines.
* When the 500 for the day are used up, the notebook saves everything, says so, and still makes all the tables from what is finished. Run all again after the limit resets (about 1–2 PM Bangladesh time) and it carries on: day 1 and day 2 finish Part 1, day 3 finishes Part 2.

**Progress is saved** after every Gemini answer, in `SafeCrop/results/loop_v2/gemini_v2_calls.csv`. If anything stops, **Run all** again and it continues without asking anything twice.

In [ ]:
# Install or update what's needed (about 30 seconds). Safe to run again.
import subprocess, sys, re
from importlib.metadata import version, PackageNotFoundError

def ensure(pkg, min_version=None):
    try:
        v = version(pkg)
        if min_version:
            have = tuple(int(x) for x in re.findall(r"\d+", v)[:3])
            if have < tuple(int(x) for x in min_version.split(".")):
                raise PackageNotFoundError(pkg)
    except PackageNotFoundError:
        print("installing", pkg, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", pkg], check=True)

ensure("google-genai", "1.30.0")
ensure("transformers")
ensure("openpyxl")
print("google-genai", version("google-genai"), "| transformers", version("transformers"))

In [ ]:
import os, io, re, csv, json, time, glob, random, shutil, zipfile, datetime, threading, subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image as ShowImage
from tqdm.auto import tqdm

def short(e, n=300):
    s = str(e).replace("\n", " ")
    return s if len(s) <= n else s[:n] + " ..."

print("pandas", pd.__version__, "| numpy", np.__version__)

## Settings

Nothing needs changing. The free Gemini key comes from the Colab secret `GEMINI_API_KEY` (see step 3 above).

In [ ]:
# You don't need to change anything here. The Gemini key is read from the Colab secret GEMINI_API_KEY.

# ---- what to run ----
RUN_CLIP = True           # CLIP photo database: runs inside Colab, free (about 5 minutes on a GPU)
RUN_GEMINI = True         # Gemini 3.5 Flash-Lite with the free key (about 1,450 questions, 500 a day)
GEMINI_MODEL = "gemini-3.5-flash-lite"

# ---- angles: each test is repeated for every view in this list ----
VIEWS = ["original", "rot90", "rot180", "mirror"]   # CLIP (free, so all angles)
GEMINI_VIEWS = ["original"]  # Gemini: original angle only (about 1,450 questions). Each extra angle adds about 1,450.
# rot90 = turned a quarter clockwise, rot180 = upside down, mirror = flipped left to right (the "other side").
# Also available: "rot45" (turned 45 degrees), "rot270" (a quarter anticlockwise) and "tilt" (camera tilted back).

# ---- the three pipelines: where each one starts, and the clearer levels it moves through ----
PIPELINES = {"P3": ["L3", "L2", "L1", "L0"],
             "P2": ["L2", "L1", "L0"],
             "P1": ["L1", "L0"]}

# ---- when a model counts as confident (it stops there) ----
CONFIDENCE_TO_STOP = 0.8  # Gemini: says "clear enough" AND confidence >= this
CLIP_STOP_SCORE = 0.8     # CLIP: measured score >= this (photo quality + CLIP's own confidence, as last week)

# ---- what is carried to the next level ----
CLIP_CARRY_WEIGHT = 0.5   # CLIP: share of the previous photo's features mixed into the new one (0.5 = half and half)
GEMINI_CARRY = "notes"    # "notes": Gemini gets its notes from the previous photo | "notes+photo": the previous photo as well

# ---- size and speed ----
PHOTOS_PER_DISEASE = None # None = all 210 photos; e.g. 10 = 70 photos (a third of the questions)
FREE_KEY_REQUESTS_PER_MINUTE = 14   # the free limit for Gemini 3.5 Flash-Lite is 15 a minute

# ---- CLIP model and images (same as before) ----
CLIP_MODEL = "openai/clip-vit-large-patch14"
CLIP_DATABASE_NEIGHBOURS = 5
SOURCE = "auto"           # "auto": look in Google Drive, ask for an upload if not found | "drive" | "upload"
DATASET_PATH = ""         # optional exact path to noisy_images.zip
SEED = 299

# ---- fixed: the free AI Studio key, no Google Cloud billing (leave these as they are) ----
BACKEND = "api_key"
GEMINI_PREFERRED = [GEMINI_MODEL]
PROJECT_ID, LOCATION = "", "global"
PARALLEL_REQUESTS, MAX_REQUESTS_PER_MINUTE = 16, 240
BUDGET_USD, PRICE_INPUT_PER_MILLION, PRICE_OUTPUT_PER_MILLION = 60, 0.0, 0.0

## 1. Get the images

This works the same as before: it finds `noisy_images.zip` in your Google Drive, or asks you to upload it. The results go to `SafeCrop/results/loop_v2` in your Drive.

In [ ]:
WORK = "/content/safecrop_loop_v2"
DATA_DIR = os.path.join(WORK, "data")
os.makedirs(DATA_DIR, exist_ok=True)

def mount_drive():
    if os.path.isdir("/content/drive/MyDrive"):
        return True
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Couldn't connect to Google Drive:", short(e, 200))
    return os.path.isdir("/content/drive/MyDrive")

def walk(top, max_depth):
    # os.walk that stops max_depth folders below `top` and skips hidden folders
    top = top.rstrip(os.sep)
    base = top.count(os.sep)
    for root, dirs, files in os.walk(top):
        depth = root.count(os.sep) - base
        dirs[:] = sorted(d for d in dirs if not d.startswith(".")) if depth < max_depth else []
        yield root, files

def is_dataset_folder(root, files):
    return "metadata.csv" in files and os.path.isdir(os.path.join(root, "L0"))

def find_dataset_folder(top):
    for root, files in walk(top, max_depth=6):
        if is_dataset_folder(root, files):
            return root
    return None

def unpack(zip_path, dest):
    with zipfile.ZipFile(zip_path) as z:
        members = [m for m in z.namelist()
                   if not m.startswith("__MACOSX") and not os.path.basename(m.rstrip("/")).startswith("._")]
        z.extractall(dest, members)

def prepare(path):
    # returns the folder that holds metadata.csv and L0..L3, unzipping first if needed
    if os.path.isdir(path):
        return find_dataset_folder(path)
    if not (os.path.isfile(path) and zipfile.is_zipfile(path)):
        return None
    dest = os.path.join(DATA_DIR, re.sub(r"[^\w.-]+", "_", os.path.splitext(os.path.basename(path))[0]))
    if os.path.exists(dest):
        shutil.rmtree(dest)
    print("Unzipping", os.path.basename(path), "...")
    unpack(path, dest)
    for _ in range(3):   # handles a zip inside the zip
        found = find_dataset_folder(dest)
        if found:
            return found
        inner = [os.path.join(r, f) for r, fs in walk(dest, 6) for f in fs if f.lower().endswith(".zip")]
        inner = [p for p in inner if "noisy" in os.path.basename(p).lower()] or inner
        if not inner:
            break
        for p in inner:
            unpack(p, os.path.splitext(p)[0])
            os.remove(p)
    return None

def find_in_drive():
    found = []
    for root, files in walk("/content/drive/MyDrive", max_depth=4):
        if is_dataset_folder(root, files):
            found.append(root)
        for f in files:
            fl = f.lower()
            if fl.endswith(".zip") and "noisy" in fl and "result" not in fl:
                found.append(os.path.join(root, f))
    return sorted(found, key=os.path.getmtime, reverse=True)

DRIVE_OK, dataset_dir, uploaded = False, None, []

if DATASET_PATH:
    if DATASET_PATH.startswith("/content/drive"):
        DRIVE_OK = mount_drive()
    dataset_dir = prepare(DATASET_PATH)
    if dataset_dir is None:
        print("No images found at DATASET_PATH:", DATASET_PATH)

if dataset_dir is None and SOURCE in ("auto", "drive"):
    DRIVE_OK = mount_drive()
    if DRIVE_OK:
        print("Searching your Google Drive for noisy_images ...")
        candidates = find_in_drive()
        for c in candidates:
            print("  found:", c)
        for c in candidates:
            dataset_dir = prepare(c)
            if dataset_dir:
                print("Using:", c)
                break
        if dataset_dir is None:
            print("No noisy_images.zip found in your Drive. (Files in 'Shared with me' only count if you add a shortcut to My Drive.)")

if dataset_dir is None and SOURCE in ("auto", "upload"):
    from google.colab import files
    print("Click 'Choose files' and pick noisy_images.zip "
          "(add gemini_v2_calls.csv too if you are continuing an earlier run without Drive).")
    os.makedirs(os.path.join(WORK, "uploads"), exist_ok=True)
    for name, data in files.upload().items():
        p = os.path.join(WORK, "uploads", name)
        with open(p, "wb") as f:
            f.write(data)
        uploaded.append(p)
    for p in uploaded:
        if p.lower().endswith(".zip"):
            dataset_dir = prepare(p)
            if dataset_dir:
                break

if dataset_dir is None:
    raise RuntimeError("No images found. Put noisy_images.zip in your Google Drive (or set SOURCE = 'upload') and run again.")

# ---- read the list of images ----
meta = pd.read_csv(os.path.join(dataset_dir, "metadata.csv"))
need = {"image", "photo_id", "disease", "level"}
if need - set(meta.columns):
    raise RuntimeError(f"metadata.csv is missing the columns {sorted(need - set(meta.columns))}")
for col in ["disease_name", "group", "origin", "noise"]:
    if col not in meta.columns:
        meta[col] = ""
meta = meta.fillna("")
meta["disease_name"] = [n if n else d for n, d in zip(meta.disease_name, meta.disease)]
meta["level"] = meta.level.astype(str)
meta["path"] = [os.path.join(dataset_dir, p) for p in meta.image]
missing = ~meta.path.map(os.path.exists)
if missing.any():
    print(f"Warning: {missing.sum()} images listed in metadata.csv are missing and will be skipped.")
    meta = meta[~missing]
meta = meta.drop_duplicates("image").sort_values(["disease", "photo_id", "level"]).reset_index(drop=True)

# ---- where results go ----
RESULTS_DIR = "/content/drive/MyDrive/SafeCrop/results" if DRIVE_OK else os.path.join(WORK, "results")
LOOP_DIR = os.path.join(RESULTS_DIR, "loop_v2")
os.makedirs(LOOP_DIR, exist_ok=True)
CALLS_FILE = os.path.join(LOOP_DIR, "gemini_v2_calls.csv")
for p in uploaded:
    b = os.path.basename(p).lower()
    if b.endswith(".csv") and "gemini_v2_calls" in b:   # continue an earlier run without Drive
        old = pd.read_csv(p, dtype=str, keep_default_na=False)
        if os.path.exists(CALLS_FILE):
            old = pd.concat([pd.read_csv(CALLS_FILE, dtype=str, keep_default_na=False), old])
        old.drop_duplicates(["model", "kind", "pipeline", "view", "photo_id", "level"], keep="first").to_csv(CALLS_FILE, index=False)
        print("Continuing from", os.path.basename(p))

print(f"\n{len(meta)} images, {meta.photo_id.nunique()} photos, {meta.disease.nunique()} diseases. Results go to {LOOP_DIR}")
display(meta.pivot_table(index=["group", "disease_name"], columns="level", values="image", aggfunc="count", fill_value=0))

## 2. The angles

Each view is the same photo at the same noise level, only turned or flipped. The pictures below show one photo at every level (columns) and every view (rows).

The photos to test are picked here. If `PHOTOS_PER_DISEASE` is set, the same random photos are used for CLIP and for Gemini, so the two can be compared.

In [ ]:
ORDER = ["L3", "L2", "L1", "L0"]          # most damaged first

def _perspective_coeffs(src, dst):
    # coefficients for Image.transform(PERSPECTIVE): output point (dst) -> input point (src)
    A, b = [], []
    for (x, y), (u, v) in zip(dst, src):
        A.append([x, y, 1, 0, 0, 0, -u * x, -u * y]); b.append(u)
        A.append([0, 0, 0, x, y, 1, -v * x, -v * y]); b.append(v)
    return np.linalg.solve(np.array(A, float), np.array(b, float)).tolist()

def make_view(img, view):
    if view == "original":
        return img
    if view == "rot90":
        return img.rotate(-90, expand=True)         # a quarter turn clockwise
    if view == "rot180":
        return img.rotate(180)                      # upside down
    if view == "rot45":                             # turned 45 degrees; the corners are filled with a mirror of the edges
        W, H = img.size
        p = int(0.3 * max(W, H))
        big = Image.fromarray(np.pad(np.asarray(img), ((p, p), (p, p), (0, 0)), mode="reflect"))
        return big.rotate(-45, resample=Image.BICUBIC).crop((p, p, p + W, p + H))
    if view == "rot270":
        return img.rotate(90, expand=True)          # a quarter turn anticlockwise
    if view == "mirror":
        return ImageOps.mirror(img)                 # flipped left to right: the leaf seen from the other side
    if view == "tilt":                              # as if the camera were tilted back (perspective)
        W, H = img.size
        src = [(0.12 * W, 0), (0.88 * W, 0), (W, H), (0, H)]
        return img.transform((W, H), Image.PERSPECTIVE, _perspective_coeffs(src, [(0, 0), (W, 0), (W, H), (0, H)]),
                             Image.BICUBIC)
    raise ValueError(f"unknown view {view!r}: use original, rot90, rot180, rot270, rot45, mirror or tilt")

def load_view(path, view):
    return make_view(Image.open(path).convert("RGB"), view)

def view_bytes(path, view):
    out = io.BytesIO()
    load_view(path, view).save(out, format="PNG")
    return out.getvalue()

ALL_VIEWS_USED = list(dict.fromkeys(VIEWS + GEMINI_VIEWS))
for v in ALL_VIEWS_USED:
    make_view(Image.new("RGB", (8, 8)), v)       # stops here with a clear message if a view name is mistyped

# ---- which photos to test (the same ones for CLIP and Gemini) ----
photo_table = meta.drop_duplicates("photo_id")[["photo_id", "disease"]]
rng = random.Random(SEED)
PHOTOS = []
for d, ids in photo_table.groupby("disease").photo_id:
    ids = sorted(ids)
    rng.shuffle(ids)
    PHOTOS += ids if PHOTOS_PER_DISEASE is None else ids[:PHOTOS_PER_DISEASE]
PHOTOS = sorted(PHOTOS)
BY_PHOTO = {pid: {r.level: r for r in g.itertuples(index=False)} for pid, g in meta.groupby("photo_id")}
PINFO = meta.drop_duplicates("photo_id").set_index("photo_id")[["disease", "disease_name", "group", "origin"]]
print(f"{len(PHOTOS)} photos x {len(ORDER)} levels: CLIP at {len(VIEWS)} angle(s) = {len(PHOTOS) * len(ORDER) * len(VIEWS)} "
      f"test images; Gemini at {len(GEMINI_VIEWS)} angle(s) = {len(PHOTOS) * len(ORDER) * len(GEMINI_VIEWS)} test images")

# ---- show one photo at every level and angle ----
example = PHOTOS[0]
fig, axes = plt.subplots(len(ALL_VIEWS_USED), len(ORDER), figsize=(2.1 * len(ORDER), 2.1 * len(ALL_VIEWS_USED)), dpi=110)
axes = np.atleast_2d(axes)
for i, v in enumerate(ALL_VIEWS_USED):
    for j, lv in enumerate(ORDER):
        ax = axes[i, j]
        if lv in BY_PHOTO[example]:
            ax.imshow(load_view(BY_PHOTO[example][lv].path, v))
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_title(lv, fontsize=10)
        if j == 0:
            ax.set_ylabel(v, fontsize=10)
fig.suptitle(f"{PINFO.loc[example, 'disease_name']} ({example}): every level (columns) at every angle (rows)", fontsize=10)
fig.tight_layout()
fig.savefig(os.path.join(LOOP_DIR, "views_example.jpg"), facecolor="white", pil_kwargs={"quality": 88})
plt.show()

## 3. The 33 answer options and the question

The options are the same as before. The question has one addition: Gemini also writes down the **visible signs** it sees, such as the spot colour, the yellowing pattern or the damaged part.

These notes are the "features" Gemini carries to the next level in a pipeline.

In [ ]:
# key, crop, name shown to Gemini (after "Crop: "), full name
LABELS = [dict(key=k, crop=c, gemini=g, clip=n) for k, c, g, n in [
    ("apple_scab",                      "apple",      "scab",                                  "apple scab"),
    ("apple_cedar_rust",                "apple",      "cedar apple rust",                      "cedar apple rust"),
    ("apple_alternaria_leaf_spot",      "apple",      "Alternaria leaf spot",                  "apple Alternaria leaf spot"),
    ("apple_powdery_mildew",            "apple",      "powdery mildew",                        "apple powdery mildew"),
    ("apple_healthy",                   "apple",      "healthy",                               "healthy apple"),
    ("potato_late_blight",              "potato",     "late blight",                           "potato late blight"),
    ("potato_early_blight",             "potato",     "early blight",                          "potato early blight"),
    ("potato_healthy",                  "potato",     "healthy",                               "healthy potato"),
    ("grape_leaf_blight",               "grape",      "leaf blight (Isariopsis leaf spot)",    "grape leaf blight"),
    ("grape_black_rot",                 "grape",      "black rot",                             "grape black rot"),
    ("grape_esca",                      "grape",      "esca (black measles)",                  "grape esca"),
    ("grape_downy_mildew",              "grape",      "downy mildew",                          "grape downy mildew"),
    ("grape_healthy",                   "grape",      "healthy",                               "healthy grape"),
    ("pear_leaf_spot",                  "pear",       "leaf spot",                             "pear leaf spot"),
    ("pear_scab",                       "pear",       "scab",                                  "pear scab"),
    ("pear_fire_blight",                "pear",       "fire blight",                           "pear fire blight"),
    ("pear_rust",                       "pear",       "rust",                                  "pear rust"),
    ("pear_healthy",                    "pear",       "healthy",                               "healthy pear"),
    ("rice_stem_borer",                 "rice",       "stem borer damage (dead heart / white head)", "rice stem borer damage"),
    ("rice_blast",                      "rice",       "blast",                                 "rice blast"),
    ("rice_brown_spot",                 "rice",       "brown spot",                            "rice brown spot"),
    ("rice_bacterial_leaf_blight",      "rice",       "bacterial leaf blight",                 "rice bacterial leaf blight"),
    ("rice_sheath_blight",              "rice",       "sheath blight",                         "rice sheath blight"),
    ("rice_healthy",                    "rice",       "healthy",                               "healthy rice"),
    ("jute_mosaic",                     "jute",       "mosaic",                                "jute mosaic"),
    ("jute_dieback",                    "jute",       "dieback",                               "jute dieback"),
    ("jute_stem_rot",                   "jute",       "stem rot",                              "jute stem rot"),
    ("jute_anthracnose",                "jute",       "anthracnose",                           "jute anthracnose"),
    ("jute_healthy",                    "jute",       "healthy",                               "healthy jute"),
    ("black_gram_yellow_mosaic",        "black gram", "yellow mosaic",                         "black gram yellow mosaic"),
    ("black_gram_cercospora_leaf_spot", "black gram", "Cercospora leaf spot",                  "black gram Cercospora leaf spot"),
    ("black_gram_powdery_mildew",       "black gram", "powdery mildew",                        "black gram powdery mildew"),
    ("black_gram_healthy",              "black gram", "healthy",                               "healthy black gram"),
]]
known = {l["key"] for l in LABELS}
for d, name in meta[["disease", "disease_name"]].drop_duplicates().itertuples(index=False):
    if d not in known:   # a disease folder that isn't in the list still works: it gets its own option
        crop = name.split()[0].lower() if name else "plant"
        LABELS.append(dict(key=d, crop=crop, gemini=name, clip=name.lower()))
        print("Added an option for", d)

LABEL_KEYS = [l["key"] for l in LABELS]
OPTIONS = [f"{l['crop'][0].upper() + l['crop'][1:]}: {l['gemini']}" for l in LABELS]   # what Gemini sees
OPTION_TO_KEY = dict(zip(OPTIONS, LABEL_KEYS))

PROMPT = (
    "You are an expert plant pathologist. The photo shows part of a crop plant. "
    "It may be blurry, dark, grainy or compressed.\n"
    "Which ONE option below best describes the main disease, pest damage or condition you can see? "
    "Pick the single most likely option, even if you are unsure.\n\nOptions:\n"
    + "\n".join("- " + o for o in OPTIONS)
    + "\n\nAnswer in JSON with \"visible_signs\" (what you can actually see in the photo, in at most 25 words: "
      "for example spot colour and shape, yellowing pattern, damaged plant part), \"diagnosis\" (exactly one option "
      "from the list), \"confidence\" (from 0 to 1) and \"clear_enough\" (true if the photo is clear enough for a "
      "reliable diagnosis, false if the farmer should take it again)."
)
FOLLOWUP = ("The farmer took the photo again. Here is a clearer photo of the same plant. Look at both photos and "
            "answer again, in the same JSON format.")

def to_float(x):
    try:
        return float(x)
    except Exception:
        return None

def to_bool(x):
    if isinstance(x, bool):
        return x
    return {"true": True, "false": False}.get(str(x).strip().lower())

def carry_note(prev_level, prev):
    # Gemini's own notes from the previous level: the "features" a pipeline carries forward
    conf = to_float(prev.get("confidence"))
    return ("Earlier, the farmer sent a lower-quality photo of this same plant. From that photo you noted:\n"
            "- visible signs: " + (prev.get("signs") or "nothing clear") + "\n"
            "- diagnosis: " + (prev.get("answer") or "no usable answer")
            + (f" (confidence {conf:.2f})" if conf is not None else "") + "\n"
            "- you were not confident enough to stop there.\n"
            "Here is a clearer photo of the same plant. Use your earlier notes together with this new photo.\n\n")

def norm_text(s):
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

NORM_OPTIONS = {norm_text(o): o for o in OPTIONS}

def to_key(answer):
    if answer in OPTION_TO_KEY:
        return OPTION_TO_KEY[answer]
    n = norm_text(answer)
    if n in NORM_OPTIONS:
        return OPTION_TO_KEY[NORM_OPTIONS[n]]
    hits = [o for k, o in NORM_OPTIONS.items() if k and k in n]
    return OPTION_TO_KEY[max(hits, key=len)] if hits else ""

def parse_reply(text):
    text = (text or "").strip()
    data = {}
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            data = json.loads(m.group(0))
        except Exception:
            data = {}
    if not isinstance(data, dict):
        data = {}
    answer = data.get("diagnosis") or text
    try:
        conf = float(data.get("confidence"))
        conf = conf / 100 if 1 < conf <= 100 else conf
        conf = min(max(conf, 0.0), 1.0)
    except Exception:
        conf = ""
    clear = data.get("clear_enough", "")
    if isinstance(clear, str):
        clear = {"true": True, "yes": True, "false": False, "no": False}.get(clear.strip().lower(), "")
    signs = data.get("visible_signs", "")
    if isinstance(signs, list):
        signs = ", ".join(str(s) for s in signs)
    signs = re.sub(r"\s+", " ", str(signs or "")).strip()[:300]
    return str(answer), to_key(answer), conf, clear, signs

print(len(OPTIONS), "answer options.")

## 4. CLIP photo database: levels alone, three pipelines, every angle

This needs no API and runs inside Colab. As before, each test photo is compared with the clean (L0) photos of all the **other** plants, and the 5 most similar ones vote. The stored photos are always upright. Only the test photo is turned or flipped, just as a farmer's photo could be.

* **Levels alone:** every photo at every level and angle is judged on its own.
* **Confident:** the measured score from last week, which combines photo quality with CLIP's own confidence. It is learned from the levels-alone results and checked on photos it didn't learn from.
* **Pipelines:** when a pipeline moves to the next level, it mixes the previous level's feature vector with the new one (`CLIP_CARRY_WEIGHT`), and the mixed vector votes.

In [ ]:
CLIP_IND, CLIP_STEPS = pd.DataFrame(), pd.DataFrame()

def quality_features(img):
    from scipy import ndimage
    a = np.asarray(img).astype(np.float64) / 255
    g = a @ [0.299, 0.587, 0.114]
    sharp = np.log(ndimage.laplace(g).var() + 1e-8)                       # sharpness
    k = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]])
    conv = ndimage.convolve(g, k)
    H, W = g.shape
    sigma = np.sqrt(np.pi / 2) * np.abs(conv[1:-1, 1:-1]).sum() / (6 * (W - 2) * (H - 2))   # noise
    lum, contrast = g.mean(), g.std()                                      # light and contrast
    mx, mn = a.max(2), a.min(2)
    sat = np.where(mx > 0, (mx - mn) / np.maximum(mx, 1e-6), 0).mean()     # colour
    exg = 2 * a[..., 1] - a[..., 0] - a[..., 2]
    mask = exg > 0.05
    cov = mask.mean()                                                      # how much of the photo is leaf
    if mask.sum() > 0:
        ys, xs = np.nonzero(mask)
        off = np.hypot(xs.mean() / W - 0.5, ys.mean() / H - 0.5) / (np.sqrt(2) / 2)
    else:
        off = 1.0
    dh = np.abs(np.diff(g, axis=1))
    block = dh[:, 7::8].mean() / (np.delete(dh, np.s_[7::8], axis=1).mean() + 1e-6)  # JPEG blockiness
    return [sharp, sigma, lum, contrast, sat, cov, off, block]

if RUN_CLIP:
    # ---- feature vectors for every image at every angle (saved, so a re-run skips this) ----
    NEEDED = list(dict.fromkeys(["original"] + VIEWS))     # the database itself is always the upright photos
    EMB_FILE = os.path.join(LOOP_DIR, "clip_features.npz")
    E = {}
    if os.path.exists(EMB_FILE):
        try:
            z = np.load(EMB_FILE)
            if str(z["model"]) == CLIP_MODEL and list(z["images"]) == list(meta.image):
                E = {v: z[v] for v in z.files if v not in ("model", "images")}
        except Exception as e:
            print("The saved CLIP features couldn't be read (" + short(e, 80) + "), so they are made again.")
    todo_views = [v for v in NEEDED if v not in E]
    if todo_views:
        import torch
        from transformers import CLIPModel
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading {CLIP_MODEL} on " + ("the GPU" if device == "cuda" else
              "the CPU (slower; Runtime > Change runtime type > T4 GPU makes it faster)") + " ...")
        clip_model = CLIPModel.from_pretrained(CLIP_MODEL)
        clip_model = (clip_model.half() if device == "cuda" else clip_model.float()).to(device).eval()
        dtype = next(clip_model.parameters()).dtype
        size = clip_model.config.vision_config.image_size
        MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3, 1, 1)
        STD = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3, 1, 1)

        def prep(img):
            w, h = img.size
            s = size / min(w, h)
            if s != 1:
                img = img.resize((max(size, round(w * s)), max(size, round(h * s))), Image.BICUBIC)
            w, h = img.size
            left, upper = (w - size) // 2, (h - size) // 2
            img = img.crop((left, upper, left + size, upper + size))
            x = torch.from_numpy(np.asarray(img).astype(np.float32) / 255.0).permute(2, 0, 1)
            return (x - MEAN) / STD

        @torch.no_grad()
        def embed(view, batch=32):
            paths, out = list(meta.path), []
            for i in tqdm(range(0, len(paths), batch), desc=f"CLIP features ({view})"):
                x = torch.stack([prep(load_view(p, view)) for p in paths[i:i + batch]]).to(device=device, dtype=dtype)
                v = clip_model.visual_projection(clip_model.vision_model(pixel_values=x).pooler_output)
                out.append(torch.nn.functional.normalize(v.float(), dim=-1).cpu())
            return torch.cat(out).numpy()

        for v in todo_views:
            E[v] = embed(v)
        np.savez(EMB_FILE, model=np.array(CLIP_MODEL), images=np.array(list(meta.image)), **E)
        del clip_model
        if device == "cuda":
            torch.cuda.empty_cache()
    else:
        print("CLIP features for every angle are already saved, so the CLIP model isn't loaded again.")

    # ---- the database: clean, upright photos; a photo never finds itself ----
    DISEASES = sorted(meta.disease.unique())
    D_INDEX = {d: i for i, d in enumerate(DISEASES)}
    ref = np.flatnonzero((meta.level == "L0").to_numpy())
    R = E["original"][ref]
    ref_pid = meta.photo_id.to_numpy()[ref]
    ref_d = np.array([D_INDEX[d] for d in meta.disease.to_numpy()[ref]])
    K = max(1, min(CLIP_DATABASE_NEIGHBOURS, len(ref) - 1))

    def vote(Q, qpid):
        S = Q @ R.T
        S[np.asarray(qpid)[:, None] == ref_pid[None, :]] = -np.inf
        nn = np.argsort(-S, axis=1)[:, :K]
        rows_ = np.arange(len(Q))
        Wt = np.zeros((len(Q), len(DISEASES)))
        for j in range(K):
            np.add.at(Wt, (rows_, ref_d[nn[:, j]]), np.maximum(S[rows_, nn[:, j]], 0) + 1e-6)
        best = Wt.argmax(1)
        return [DISEASES[b] for b in best], Wt[rows_, best] / Wt.sum(1)

    # ---- levels alone: every photo, level and angle on its own ----
    sel = np.flatnonzero(meta.photo_id.isin(PHOTOS).to_numpy())
    F, rows = {}, []
    for v in VIEWS:
        F[v] = np.zeros((len(meta), 8))
        for i in tqdm(sel, desc=f"photo quality ({v})"):
            F[v][i] = quality_features(load_view(meta.path[i], v))
        pred, conf = vote(E[v][sel], meta.photo_id.to_numpy()[sel])
        for i, p_, c_ in zip(sel, pred, conf):
            r = meta.iloc[i]
            rows.append(dict(model="CLIP database", view=v, photo_id=r.photo_id, disease=r.disease,
                             disease_name=r.disease_name, group=r.group, level=r.level, idx=i, pred=p_, conf=float(c_)))
    CI = pd.DataFrame(rows)
    CI["correct"] = CI.pred == CI.disease

    # ---- measured score, learned on 4 of 5 groups of photos and used on the 5th (all levels and angles of a photo stay together) ----
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    X = np.column_stack([np.array([F[v][i] for v, i in zip(CI.view, CI.idx)]), CI.conf.to_numpy()])
    y = CI.correct.to_numpy().astype(int)
    order_ = list(dict.fromkeys(CI.photo_id))
    FOLD = {pid: (len(order_) - 1 - n) % 5 for n, pid in enumerate(order_)}
    folds = CI.photo_id.map(FOLD).to_numpy()
    QMODEL, score = {}, CI.conf.to_numpy().copy()
    for f in range(5):
        tr, te = folds != f, folds == f
        if te.any() and len(set(y[tr])) == 2:
            QMODEL[f] = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X[tr], y[tr])
            score[te] = QMODEL[f].predict_proba(X[te])[:, 1]
    CI["score"] = score
    CI["confident"] = CI.score >= CLIP_STOP_SCORE

    def measured(pid, feats, conf):
        m = QMODEL.get(FOLD.get(pid))
        return float(m.predict_proba(np.r_[feats, conf][None, :])[0, 1]) if m is not None else float(conf)

    # ---- the three pipelines, with the previous level's features mixed in ----
    lookup = {(r.view, r.photo_id, r.level): r for r in CI.itertuples(index=False)}
    idx_of = {(r.photo_id, r.level): i for i, r in enumerate(meta.itertuples(index=False))}
    steps = []
    for v in VIEWS:
        for P, levels in PIPELINES.items():
            for pid in PHOTOS:
                info = PINFO.loc[pid]
                prev_level = None
                for k, level in enumerate([l for l in levels if (pid, l) in idx_of], 1):
                    if k == 1:
                        b = lookup[(v, pid, level)]
                        pred, conf, sc, carried = b.pred, b.conf, b.score, False
                    else:
                        i, j = idx_of[(pid, level)], idx_of[(pid, prev_level)]
                        q = (1 - CLIP_CARRY_WEIGHT) * E[v][i] + CLIP_CARRY_WEIGHT * E[v][j]
                        pr, cf = vote((q / np.linalg.norm(q))[None, :], [pid])
                        pred, conf, carried = pr[0], float(cf[0]), True
                        sc = measured(pid, F[v][i], conf)
                    steps.append(dict(model="CLIP database", pipeline=P, view=v, photo_id=pid, disease=info.disease,
                                      disease_name=info.disease_name, group=info.group, step=k, level=level,
                                      carried=carried, carried_from=prev_level if carried else "", pred=pred,
                                      score=float(sc), confident=bool(sc >= CLIP_STOP_SCORE),
                                      correct=pred == info.disease))
                    if sc >= CLIP_STOP_SCORE:
                        break
                    prev_level = level
    CLIP_STEPS = pd.DataFrame(steps)
    CLIP_IND = CI.drop(columns=["idx"])
    CLIP_IND.to_csv(os.path.join(LOOP_DIR, "clip_levels_alone.csv"), index=False)
    CLIP_STEPS.to_csv(os.path.join(LOOP_DIR, "clip_pipeline_steps.csv"), index=False)
    print("\nCLIP database, accuracy (%) of each level on its own, by angle:")
    display((CLIP_IND.pivot_table(index="view", columns="level", values="correct", aggfunc="mean") * 100)
            .reindex(columns=ORDER).round(1))
else:
    print("CLIP is switched off in Settings (RUN_CLIP = False).")

## 5. Connect to Gemini

This reads your free key from the Colab secret `GEMINI_API_KEY` and sends one tiny test question to Gemini 3.5 Flash-Lite. If something is wrong, it says what to fix.

In [ ]:
from google import genai
from google.genai import types, errors as genai_errors
try:
    from google.auth import exceptions as gauth_errors
    AUTH_ERRORS = (gauth_errors.GoogleAuthError,)
except Exception:
    AUTH_ERRORS = ()

class Fatal(Exception): pass        # a setup problem: stop and fix it
class DailyLimit(Exception): pass   # the quota is used up for now
class EmptyReply(Exception): pass   # Gemini sent no text
class NotFound(Exception): pass     # this model isn't available here
class Stopped(Exception): pass      # the run was stopped

STOP = threading.Event()
TIMEOUT_MS = 180_000

def sh(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=90).stdout.strip()
    except Exception:
        return ""

def advice(code, msg):
    m = msg.lower()
    pid = PROJECT_ID or "your project"
    if BACKEND != "vertex":
        return ("Gemini rejected the API key: " + short(msg, 200) + "\nCheck the GEMINI_API_KEY secret "
                "(key icon on the left), then run this cell again.")
    if "service_disabled" in m or "has not been used" in m or "it is disabled" in m:
        return (f"The Agent Platform API is not switched on for project {pid}.\n"
                f"Open https://console.cloud.google.com/apis/enableflow?apiid=aiplatform.googleapis.com&project={pid} , "
                "click Enable, wait 2 minutes, then run this cell again.")
    if "billing" in m:
        return (f"Billing is not active for project {pid}: the free trial isn't started, or it isn't linked to this project.\n"
                f"Open https://console.cloud.google.com/billing/linkedaccount?project={pid} , link the free-trial "
                "billing account, then run this cell again.")
    if code == 403 or "permission" in m:
        return (f"The Google account you signed in with can't use project {pid}.\n"
                "1) Check that PROJECT_ID in Settings is the project's ID (not its name).\n"
                "2) If it is right, sign in again with the account that created the project: "
                "Runtime > Disconnect and delete runtime, then Runtime > Run all.")
    if code == 401 or "unauthenticated" in m or "credential" in m:
        return "Google Cloud sign-in didn't work. Run this cell again and finish the sign-in window."
    return "Google Cloud said: " + short(msg, 400)

def retry_seconds(msg):
    m = re.search(r"retryDelay\W+(\d+(?:\.\d+)?)s", msg) or re.search(r"retry in (\d+(?:\.\d+)?)\s*s", msg, re.I)
    return float(m.group(1)) if m else None

class Pacer:
    # spaces requests out across all threads. It slows down when Google says "too many requests" (several
    # threads hitting the same busy moment count once) and speeds up again after a run of successes.
    def __init__(self, per_minute):
        self.top = float(per_minute)
        self.rate = float(per_minute)
        self.lock = threading.Lock()
        self.next_at = 0.0
        self.ok = 0
        self.busy = 0
        self.last_cut = 0.0
    def wait(self):
        with self.lock:
            now = time.time()
            at = max(now, self.next_at)
            self.next_at = at + 60.0 / self.rate
        if at > now and STOP.wait(at - now):
            raise Stopped()
    def success(self):
        with self.lock:
            self.ok += 1
            self.busy = 0
            if self.ok >= 10 and self.rate < self.top:
                self.rate = min(self.top, self.rate * 1.25)
                self.ok = 0
    def too_many(self):
        with self.lock:
            self.busy += 1
            now = time.time()
            if now - self.last_cut >= 5:
                self.rate = max(6.0, self.rate * 0.7)
                self.last_cut = now
                self.ok = 0
            self.next_at = max(self.next_at, now + 60.0 / self.rate)
            return self.busy

def nap(seconds):
    if STOP.wait(seconds):
        raise Stopped()

SCHEMA = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "visible_signs": types.Schema(type=types.Type.STRING),
        "diagnosis": types.Schema(type=types.Type.STRING, enum=OPTIONS),
        "confidence": types.Schema(type=types.Type.NUMBER),
        "clear_enough": types.Schema(type=types.Type.BOOLEAN),
    },
    required=["visible_signs", "diagnosis", "confidence", "clear_enough"],
    property_ordering=["visible_signs", "diagnosis", "confidence", "clear_enough"],
)
FORMAT = [2]        # 2 = strict answer schema, 1 = JSON only, 0 = plain text
LOCK_FMT = threading.Lock()

def ask(client, model, contents, plain=False, fmt=2):
    if plain or fmt == 0:
        config = None
    elif fmt == 2:
        config = types.GenerateContentConfig(response_mime_type="application/json", response_schema=SCHEMA)
    else:
        config = types.GenerateContentConfig(response_mime_type="application/json")
    resp = client.models.generate_content(model=model, contents=contents, config=config)
    try:
        text = resp.text
    except Exception:
        text = None
    u = getattr(resp, "usage_metadata", None)
    usage = {k: (getattr(u, k, None) or 0) for k in
             ("prompt_token_count", "candidates_token_count", "thoughts_token_count")} if u is not None else {}
    if not text:
        why = ""
        try:
            why = str(resp.candidates[0].finish_reason)
        except Exception:
            pass
        raise EmptyReply(why or "empty reply")
    return text, usage

def call_gemini(contents, client=None, model=None, attempts=8, plain=False):
    client = client or CLIENT
    model = model or GEM_MODEL
    last = None
    for attempt in range(attempts):
        if STOP.is_set():
            raise Stopped()
        PACER.wait()
        fmt = FORMAT[0]
        try:
            out = ask(client, model, contents, plain=plain, fmt=fmt)
            PACER.success()
            return out
        except genai_errors.APIError as e:
            code, msg, last = getattr(e, "code", None), str(e), e
            if code == 429:
                if re.search(r"per.?day|daily", msg, re.I):   # e.g. ..._requests_per_day_... = the daily quota
                    raise DailyLimit(msg)
                if PACER.too_many() >= BUSY_LIMIT:
                    raise DailyLimit(msg)
                nap((retry_seconds(msg) or min(60, 4 * 2 ** attempt)) + random.uniform(0, 3))
                continue
            if code == 404:
                raise NotFound(msg)
            if code in (401, 403):
                raise Fatal(advice(code, msg))
            if code == 400:
                if not plain and fmt > 0 and re.search(r"schema|enum|mime|json", msg, re.I) and attempt < attempts - 1:
                    with LOCK_FMT:
                        if FORMAT[0] == fmt:
                            FORMAT[0] = fmt - 1
                            print("  (the model refused the " + ("strict answer format; switching to plain JSON answers)"
                                  if fmt == 2 else "JSON format; switching to plain-text answers, read the same way)"))
                    continue
                if BACKEND != "vertex" and re.search(r"api.?key", msg, re.I):
                    raise Fatal(advice(code, msg))
                raise
            if code is None or code >= 500 or code in (408, 409):
                nap(min(60, 4 * 2 ** attempt) + random.uniform(0, 2))
                continue
            raise
        except EmptyReply as e:
            last = e
            if attempt >= 1:
                raise
        except (Fatal, DailyLimit, NotFound, Stopped):
            raise
        except AUTH_ERRORS as e:
            raise Fatal(advice(401, str(e)))
        except Exception as e:      # a network hiccup or a timeout
            last = e
            nap(min(60, 4 * 2 ** attempt))
    raise RuntimeError(f"gave up after {attempts} attempts: {short(last, 200)}")

def get_api_key():
    for name in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):
        try:
            from google.colab import userdata
            k = userdata.get(name)
            if k:
                return k.strip()
        except Exception:
            pass
        if os.environ.get(name):
            return os.environ[name].strip()
    from getpass import getpass
    return getpass("Paste your Gemini API key here and press Enter: ").strip()

def stop_here(message):
    print("\n" + "=" * 70 + "\n⚠️  " + message + "\n" + "=" * 70)
    raise RuntimeError("Stopped: fix the problem described above, then run this cell again.")

# ---- sign in (only when Gemini is switched on) ----
CLIENT, GEM_MODEL, PRETTY_MODEL = None, None, None
if RUN_GEMINI:
    STOP.clear()
    if BACKEND == "vertex":
        try:
            from google.colab import auth as colab_auth
            print("Signing in to Google Cloud. In the window that opens, choose the account that owns your Cloud project.")
            colab_auth.authenticate_user()
        except ImportError:
            print("(Not in Colab: using this computer's Google Cloud login.)")
        account = sh(["gcloud", "config", "get-value", "account"])
        if account:
            print("Signed in as", account)
        if not PROJECT_ID:
            ids = [l.split()[0] for l in sh(["gcloud", "projects", "list", "--format=value(projectId)"]).splitlines() if l.strip()]
            if len(ids) == 1:
                PROJECT_ID = ids[0]
                print("Using your only project:", PROJECT_ID)
            else:
                if ids:
                    print("Your projects:", ", ".join(ids))
                PROJECT_ID = input("Type the Project ID to use and press Enter: ").strip()
        PROJECT_ID = PROJECT_ID.strip()
        if not PROJECT_ID:
            stop_here("No Project ID. Put it in PROJECT_ID in the Settings cell (Cloud console > project picker > ID column).")
        locations = [LOCATION] + [l for l in ("global", "us-central1") if l != LOCATION]
        PACER, BUSY_LIMIT, WORKERS = Pacer(MAX_REQUESTS_PER_MINUTE), 40, max(1, int(PARALLEL_REQUESTS))
        def make_client(loc):
            return genai.Client(vertexai=True, project=PROJECT_ID, location=loc,
                                http_options=types.HttpOptions(timeout=TIMEOUT_MS))
    else:
        API_KEY = get_api_key()
        if not API_KEY:
            stop_here("No API key. Add the GEMINI_API_KEY secret (key icon on the left) or use BACKEND = 'vertex'.")
        locations = [None]
        PACER, BUSY_LIMIT, WORKERS = Pacer(FREE_KEY_REQUESTS_PER_MINUTE or 4), 8, 6
        def make_client(loc):
            return genai.Client(api_key=API_KEY, http_options=types.HttpOptions(timeout=TIMEOUT_MS))

    # ---- which model: keep the model of an earlier run, so the answers stay comparable ----
    earlier = None
    if os.path.exists(CALLS_FILE):
        c = pd.read_csv(CALLS_FILE, dtype=str, keep_default_na=False)
        if len(c):
            earlier = c.model.value_counts().index[0]
    if GEMINI_MODEL != "auto":
        candidates = [GEMINI_MODEL]
    else:
        candidates = ([earlier] if earlier else []) + [m for m in GEMINI_PREFERRED if m != earlier]

    CLIENT, GEM_MODEL, LOC_USED = None, None, None
    print("Finding a Gemini model that works with one tiny test question ...")
    try:
        for loc in locations:
            client = make_client(loc)
            for name in candidates:
                try:
                    call_gemini("Reply with the single word OK.", client=client, model=name, attempts=4, plain=True)
                    CLIENT, GEM_MODEL, LOC_USED = client, name, loc
                    print(f"  ✓ {name} works" + (f" (location: {loc})" if loc else ""))
                    break
                except NotFound:
                    print(f"  ✗ {name}: not available" + (f" in {loc}" if loc else ""))
                except DailyLimit as e:
                    print(f"  ✗ {name}: Google says 'too many requests' ({short(e, 120)})")
                except Fatal:
                    raise
                except Exception as e:
                    print(f"  ✗ {name}: {short(e, 160)}")
            if CLIENT is not None:
                break
    except Fatal as e:
        stop_here(str(e))
    if CLIENT is None:
        try:
            names = sorted(m.name.split("/")[-1] for m in client.models.list()
                           if "gemini" in m.name and "generateContent" in (getattr(m, "supported_actions", None) or ["generateContent"]))
            print("Models this key/project can use:", ", ".join(names))
        except Exception:
            pass
        stop_here("No Gemini model answered. If it said 'not available', put the exact name from the list above in GEMINI_MODEL. "
                  "If Google said 'too many requests', wait a few minutes and run this cell again.")
    if earlier and GEM_MODEL != earlier:
        print(f"Note: the earlier answers came from {earlier}; this run uses {GEM_MODEL}, so it starts fresh.")
    if BACKEND != "vertex":       # free key: go just under this model's per-minute limit
        PACER.top = PACER.rate = float(FREE_KEY_REQUESTS_PER_MINUTE or 14)
        print(f"Free key: up to {PACER.top:g} questions a minute.")
    PRETTY_MODEL = GEM_MODEL.replace("gemini-", "Gemini ").replace("-flash", " Flash").replace("-lite", "-Lite")
    print(f"\nReady: {GEM_MODEL} through " + (f"Google Cloud (project {PROJECT_ID}, {LOC_USED})" if BACKEND == "vertex" else "the AI Studio key")
          + f", {WORKERS} questions at a time.")
else:
    print("Gemini is switched off in Settings (RUN_GEMINI = False).")

## 6. Gemini: levels alone, three pipelines, every angle

For each photo and angle, this:
1. asks about **L3, L2, L1 and L0 separately**. These are fresh questions with one photo each, so nothing carries over. They give the levels-alone accuracy;
2. runs **P3, P2 and P1**. The first step of each pipeline is the levels-alone answer at its start level, so it costs nothing extra. Each later step sends the new photo together with Gemini's notes from the level just before.

Photos and angles are worked on in a random order, so a part-finished run still covers every disease and angle evenly.

To stop early, click the stop button (■) on this cell. When it prints *Stopped*, run the cells under **7. Results** one by one with their ▶ buttons. The tables then use the finished part.

In [ ]:
G_COLUMNS = ["model", "kind", "pipeline", "view", "photo_id", "disease", "level", "carried_from", "answer",
             "pred_label", "confidence", "clear_enough", "signs", "raw", "prompt_tokens", "output_tokens",
             "thinking_tokens", "seconds", "time"]
LOCK = threading.Lock()
EMPTY = json.dumps({"visible_signs": "", "diagnosis": "", "confidence": 0, "clear_enough": False})

def load_calls():
    if not os.path.exists(CALLS_FILE):
        return {}
    df = pd.read_csv(CALLS_FILE, dtype=str, keep_default_na=False)
    out = {}
    for r in df.to_dict("records"):
        out.setdefault((r["model"], r["kind"], r["pipeline"], r["view"], r["photo_id"], r["level"]), r)
    return out

CACHE = load_calls()

class OverBudget(Exception): pass    # the cost estimate reached BUDGET_USD

def call_cost(rec):
    # estimated US$ for one answer, from the tokens Google reported for it
    t = lambda c: to_float(rec.get(c)) or 0.0
    return (t("prompt_tokens") * PRICE_INPUT_PER_MILLION
            + (t("output_tokens") + t("thinking_tokens")) * PRICE_OUTPUT_PER_MILLION) / 1e6

COST = {"spent": 0.0, "answers": 0}
for _k, _r in CACHE.items():
    if _k[0] == GEM_MODEL:
        COST["spent"] += call_cost(_r)
        COST["answers"] += 1

def is_confident(rec, threshold=None):
    t = CONFIDENCE_TO_STOP if threshold is None else threshold
    c = to_float(rec.get("confidence"))
    return to_bool(rec.get("clear_enough")) is True and rec.get("pred_label", "") != "" and c is not None and c >= t

def save_call(rec):
    rec = {k: ("" if v is None else str(v)) for k, v in rec.items()}
    key = (rec["model"], rec["kind"], rec["pipeline"], rec["view"], rec["photo_id"], rec["level"])
    with LOCK:
        if key in CACHE:
            return CACHE[key]
        new = not os.path.exists(CALLS_FILE)
        with open(CALLS_FILE, "a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=G_COLUMNS)
            if new:
                w.writeheader()
            w.writerow({c: rec.get(c, "") for c in G_COLUMNS})
        CACHE[key] = rec
        COST["spent"] += call_cost(rec)
        COST["answers"] += 1
        return rec

def image_part(path, view):
    return types.Part.from_bytes(data=view_bytes(path, view), mime_type="image/png")

def ask_and_save(kind, pipeline, pid, view, level, carried_from, contents):
    if BACKEND == "vertex" and COST["spent"] >= BUDGET_USD:
        raise OverBudget()
    t0 = time.time()
    try:
        text, usage = call_gemini(contents)
    except EmptyReply:
        text, usage = "", {}           # no usable answer: counts as "not confident"
    answer, label, conf, clear, signs = parse_reply(text)
    return save_call({
        "model": GEM_MODEL, "kind": kind, "pipeline": pipeline, "view": view, "photo_id": pid,
        "disease": PINFO.loc[pid, "disease"], "level": level, "carried_from": carried_from, "answer": answer[:300],
        "pred_label": label, "confidence": conf, "clear_enough": clear, "signs": signs, "raw": (text or "")[:2000],
        "prompt_tokens": usage.get("prompt_token_count", ""), "output_tokens": usage.get("candidates_token_count", ""),
        "thinking_tokens": usage.get("thoughts_token_count", ""), "seconds": round(time.time() - t0, 1),
        "time": datetime.datetime.now().isoformat(timespec="seconds")})

def g_alone(pid, view, level):
    rec = CACHE.get((GEM_MODEL, "alone", "", view, pid, level))
    if rec is not None:
        return rec
    return ask_and_save("alone", "", pid, view, level, "", [PROMPT, image_part(BY_PHOTO[pid][level].path, view)])

def g_carry(P, pid, view, level, prev_level, prev):
    rec = CACHE.get((GEM_MODEL, "carry", P, view, pid, level))
    if rec is not None:
        return rec
    path = BY_PHOTO[pid][level].path
    if GEMINI_CARRY == "notes+photo":
        contents = [types.Content(role="user", parts=[types.Part(text=PROMPT), image_part(BY_PHOTO[pid][prev_level].path, view)]),
                    types.Content(role="model", parts=[types.Part(text=prev.get("raw") or EMPTY)]),
                    types.Content(role="user", parts=[types.Part(text=FOLLOWUP), image_part(path, view)])]
    else:
        contents = [carry_note(prev_level, prev) + PROMPT, image_part(path, view)]
    return ask_and_save("carry", P, pid, view, level, prev_level, contents)

def run_task(pid, view, alone_only=False):
    alone = {}
    for level in ORDER:                                   # 1) every level on its own
        if level in BY_PHOTO[pid]:
            if STOP.is_set():
                raise Stopped()
            alone[level] = g_alone(pid, view, level)
    if alone_only:
        return
    for P, levels in PIPELINES.items():                   # 2) the pipelines
        prev, prev_level = None, None
        for k, level in enumerate([l for l in levels if l in BY_PHOTO[pid]], 1):
            if STOP.is_set():
                raise Stopped()
            rec = alone[level] if k == 1 else g_carry(P, pid, view, level, prev_level, prev)
            if is_confident(rec):
                break
            prev, prev_level = rec, level

def pipeline_from_cache(P, pid, view, threshold=None):
    # replays a pipeline from saved answers: [(level, answer, carried?), ...], or None if an answer is missing
    seq = []
    for k, level in enumerate([l for l in PIPELINES[P] if l in BY_PHOTO[pid]], 1):
        rec = CACHE.get((GEM_MODEL, "alone", "", view, pid, level) if k == 1 else (GEM_MODEL, "carry", P, view, pid, level))
        if rec is None:
            return None
        seq.append((level, rec, k > 1))
        if is_confident(rec, threshold):
            break
    return seq

def task_done(pid, view):
    if any((GEM_MODEL, "alone", "", view, pid, l) not in CACHE for l in ORDER if l in BY_PHOTO[pid]):
        return False
    return all(pipeline_from_cache(P, pid, view) is not None for P in PIPELINES)

def log_error(pid, view, e):
    path = os.path.join(LOOP_DIR, "gemini_v2_errors.csv")
    with LOCK:
        new = not os.path.exists(path)
        with open(path, "a", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            if new:
                w.writerow(["model", "photo_id", "view", "error", "time"])
            w.writerow([GEM_MODEL, pid, view, short(e, 500), datetime.datetime.now().isoformat(timespec="seconds")])

VIEW_SEED = {v: i for i, v in enumerate(["original", "rot90", "rot180", "mirror", "rot270", "rot45", "tilt"])}

def gemini_task_order():
    # each angle has its own fixed random order of photos, and the angles take turns
    orders = {}
    for v in GEMINI_VIEWS:
        o = list(PHOTOS)
        random.Random(SEED + 2 + VIEW_SEED.get(v, 7)).shuffle(o)
        orders[v] = o
    return [(orders[v][i], v) for i in range(len(PHOTOS)) for v in GEMINI_VIEWS]

def finished_pairs():
    # photo-angle pairs whose pipelines can be reported: the ones Part 2 has reached, in its own order.
    # (Before Part 2 reaches a pair, only the pipelines that were confident at the first try look finished;
    #  counting those would show only the easy photos.)
    order, ok = gemini_task_order(), set()
    for v in GEMINI_VIEWS:
        for t in [t for t in order if t[1] == v]:
            if not task_done(*t):
                break
            ok.add(t)
    return ok

def alone_done(pid, view):
    return all((GEM_MODEL, "alone", "", view, pid, l) in CACHE for l in ORDER if l in BY_PHOTO[pid])

def run_phase(todo, alone_only, desc):
    STOP.clear()
    status, detail, failed, in_a_row = "done", "", 0, 0
    bar = tqdm(total=len(todo), desc=desc)
    ex = ThreadPoolExecutor(max_workers=WORKERS)
    futures = {ex.submit(run_task, pid, v, alone_only): (pid, v) for pid, v in todo}
    try:
        for fut in as_completed(futures):
            pid, v = futures[fut]
            try:
                fut.result()
                in_a_row = 0
            except Stopped:
                pass
            except OverBudget:
                if status == "done":
                    status = "budget"
                STOP.set()
            except DailyLimit as e:
                if status == "done":
                    status, detail = "limit", short(e, 300)
                STOP.set()
            except Fatal as e:
                if status == "done":
                    status, detail = "fatal", str(e)
                STOP.set()
            except Exception as e:
                failed += 1
                in_a_row += 1
                log_error(pid, v, e)
                if in_a_row >= 10 and status == "done":
                    status, detail = "errors", short(e, 300)
                    STOP.set()
            bar.update(1)
    except KeyboardInterrupt:
        status = "stopped"
        STOP.set()
        print("\nStopping: waiting for the questions already on their way ...")
    finally:
        ex.shutdown(wait=True, cancel_futures=True)
        bar.close()
    return status, detail, failed

def run_all():
    tasks = gemini_task_order()
    todo1 = [t for t in tasks if not alone_done(*t)]
    todo = [t for t in tasks if not task_done(*t)]
    guess = len(todo1) * len(ORDER) + round(len(todo) * 2.8)   # levels alone + about 2.8 pipeline steps per pair
    per_answer = COST["spent"] / COST["answers"] if COST["answers"] >= 20 and COST["spent"] > 0 else 0.005
    sec = max(10 / max(WORKERS, 1), 60 / PACER.top)            # about 10 s per answer, or the speed limit
    print(f"=== Gemini ({GEM_MODEL}): {len(tasks)} photo-angle pairs, {len(tasks) - len(todo)} already done, {len(todo)} to go ===")
    if not todo:
        return "done", ""
    print(f"About {guess:,} questions left, roughly {guess * sec / 60:.0f} minutes"
          + (f" and ${guess * per_answer:,.0f} more (so far: ${COST['spent']:,.2f} of the ${BUDGET_USD:g} budget)."
             if BACKEND == "vertex" else " (if the daily limit allows)."))
    # part 1: every level on its own, for every photo (this is "test all the images")
    status, detail, failed = "done", "", 0
    if todo1:
        print(f"Part 1 of 2: each level on its own ({len(todo1) * len(ORDER):,} questions at most)")
        status, detail, failed = run_phase(todo1, True, "Gemini part 1: levels alone (photo-angle pairs)")
    # part 2: the three pipelines (only the steps where features are carried need new questions)
    if status == "done":
        todo = [t for t in tasks if not task_done(*t)]
        if todo:
            print(f"Part 2 of 2: the three pipelines")
            status, detail, f2 = run_phase(todo, False, "Gemini part 2: pipelines (photo-angle pairs)")
            failed += f2
    n_alone = sum(alone_done(*t) for t in tasks)
    done = sum(task_done(*t) for t in tasks)
    print(f"Levels alone finished for {n_alone} of {len(tasks)} photo-angle pairs; pipelines finished for {done}."
          + (f" {failed} hit an error and will be retried next time." if failed else ""))
    if status == "budget":
        print(f"The estimated cost of Gemini's answers reached BUDGET_USD (${BUDGET_USD:g}), so the run paused. To continue, "
              "raise BUDGET_USD in Settings and choose Runtime > Run all (finished answers are kept). Or run the "
              "cells below for tables from the finished part.")
    elif status == "limit":
        if BACKEND == "api_key":
            print("The free key's limit is used up for now (daily limits reset at midnight US Pacific time, about "
                  "1-2 PM in Bangladesh). Everything so far is saved. Run all again after that and it continues "
                  "where it stopped. The cells below make tables from the finished part. Last message:", detail)
        else:
            print("Google keeps answering 'too many requests' (or a daily quota is used up). Progress is saved: run this "
                  "cell again in a while and it continues. Last message:", detail)
    elif status == "fatal":
        print("\n⚠️ " + detail)
    elif status == "errors":
        print("10 in a row failed, so the run paused. Last error:", detail, "\nRun this cell again to continue.")
    elif status == "stopped":
        print("Stopped. Run this cell again to continue, or run the cells below for tables from the finished part.")
    return status, detail

if RUN_GEMINI and GEM_MODEL:
    STATUS = run_all()
    calls = pd.read_csv(CALLS_FILE, dtype=str, keep_default_na=False) if os.path.exists(CALLS_FILE) else pd.DataFrame()
    if len(calls):
        mine = calls[calls.model == GEM_MODEL]
        tok = {c: pd.to_numeric(mine[c], errors="coerce").fillna(0).sum() for c in ["prompt_tokens", "output_tokens", "thinking_tokens"]}
        print(f"\nAnswers saved for {GEM_MODEL}: {len(mine):,} | tokens: {tok['prompt_tokens']:,.0f} in, "
              f"{tok['output_tokens']:,.0f} out, {tok['thinking_tokens']:,.0f} thinking.")
        if BACKEND == "vertex":
            print(f"Estimated cost of all these answers: ${COST['spent']:,.2f} (budget ${BUDGET_USD:g}). "
                  "The exact amount is in the Cloud console: Billing > Reports (it updates a day later).")
else:
    print("Gemini is switched off or not connected, so this section is skipped.")

## 7. Results

The tables answer one question each:

| Table | Question it answers |
|---|---|
| **1. Each level on its own** | How good is the model at each level, with no help from other levels? Does the angle matter? |
| **2. The three pipelines** | Where does each pipeline stop, how many tries does it need, and how often is it right? The "nothing carried" columns show the same pipeline when each level is judged alone (last week's loop). |
| **3. Effect of carrying features** | At each level, are answers with the previous level's features better or worse than the same photos judged alone? Does the model just repeat its earlier guess? |
| **4. Angles** | Accuracy for each angle, at each level and for each pipeline |
| **5. Info-rich vs info-scarce** | The same numbers for group A and group B, which is our research question |

**Words used:**
* **Accuracy** is the share of tests where the model named the right disease.
* **Confident** means the model stopped there:
  * Gemini says "clear enough" with a confidence of at least 0.8;
  * CLIP's measured score is at least 0.8.
* **Confident but wrong** means it stopped, but the answer was wrong.
* **Same answer at every angle** is the share of photos where all the angles got the same answer.

In [ ]:
# ---------- gather every answer from both models into the same three tables of records ----------
GEM_PAIRS = (0, 0)
def gemini_frames(threshold=None):
    global GEM_PAIRS
    ind, steps = [], []
    if not (RUN_GEMINI and GEM_MODEL):
        return pd.DataFrame(), pd.DataFrame()
    ok_pairs = finished_pairs()
    GEM_PAIRS = (len(ok_pairs), len(PHOTOS) * len(GEMINI_VIEWS))
    for pid in PHOTOS:
        info = PINFO.loc[pid]
        for v in GEMINI_VIEWS:
            for level in ORDER:
                rec = CACHE.get((GEM_MODEL, "alone", "", v, pid, level))
                if rec is not None:
                    ind.append(dict(model=PRETTY_MODEL, view=v, photo_id=pid, disease=info.disease, disease_name=info.disease_name,
                                    group=info.group, level=level, pred=rec.get("pred_label", ""),
                                    score=to_float(rec.get("confidence")), confident=is_confident(rec, threshold),
                                    correct=rec.get("pred_label", "") == info.disease))
            for P in PIPELINES:
                if (pid, v) not in ok_pairs:
                    continue
                seq = pipeline_from_cache(P, pid, v, threshold)
                if seq is None:
                    continue
                prev_level = None
                for k, (level, rec, carried) in enumerate(seq, 1):
                    steps.append(dict(model=PRETTY_MODEL, pipeline=P, view=v, photo_id=pid, disease=info.disease,
                                      disease_name=info.disease_name, group=info.group, step=k, level=level,
                                      carried=carried, carried_from=prev_level if carried else "",
                                      pred=rec.get("pred_label", ""), score=to_float(rec.get("confidence")),
                                      confident=is_confident(rec, threshold), correct=rec.get("pred_label", "") == info.disease))
                    prev_level = level
    return pd.DataFrame(ind), pd.DataFrame(steps)

G_IND, G_STEPS = gemini_frames()
if len(G_IND):
    n_img = len(G_IND)
    print(f"{PRETTY_MODEL}: {n_img} of {len(PHOTOS) * len(ORDER) * len(GEMINI_VIEWS)} images answered on their own; "
          f"pipelines finished for {GEM_PAIRS[0]} of {GEM_PAIRS[1]} photo-angle pairs."
          + ("" if GEM_PAIRS[0] == GEM_PAIRS[1] else " Tables 2 and 3 use only the finished pairs; the rest come when Part 2 ends."))
IND = pd.concat([d for d in (CLIP_IND, G_IND) if len(d)], ignore_index=True) if (len(CLIP_IND) or len(G_IND)) else pd.DataFrame()
STEPS = pd.concat([d for d in (CLIP_STEPS, G_STEPS) if len(d)], ignore_index=True) if (len(CLIP_STEPS) or len(G_STEPS)) else pd.DataFrame()
if not len(IND):
    raise RuntimeError("No results yet: run section 4 (CLIP) and/or section 6 (Gemini) first.")
MODELS_ = list(dict.fromkeys(IND.model))
PIPES_ = list(PIPELINES)

# one row per pipeline run (a photo at one angle through one pipeline)
def runs_from(steps):
    out = []
    for (m, P, v, pid), g in steps.groupby(["model", "pipeline", "view", "photo_id"], sort=False):
        g = g.sort_values("step")
        last, first = g.iloc[-1], g.iloc[0]
        out.append(dict(model=m, pipeline=P, view=v, photo_id=pid, disease=last.disease, disease_name=last.disease_name,
                        group=last.group, tries=len(g), stopped_at=last.level if last.confident else "never",
                        confident=bool(last.confident), correct=bool(last.correct), first_correct=bool(first.correct)))
    return pd.DataFrame(out)

RUNS = runs_from(STEPS) if len(STEPS) else pd.DataFrame()

# the same pipelines with NOTHING carried: walk the levels-alone answers and stop at the first confident one
# (this is last week's loop). Comparing it with RUNS shows what carrying features does to the whole pipeline.
def runs_without_carry(ind):
    out = []
    for (m, v, pid), g in ind.groupby(["model", "view", "photo_id"], sort=False):
        by = {r.level: r for r in g.itertuples(index=False)}
        for P, levels in PIPELINES.items():
            tries, last = 0, None
            for lv in [l for l in levels if l in BY_PHOTO[pid]]:
                r = by.get(lv)
                if r is None:            # a level not asked yet (part-finished run): leave this one out
                    last = None
                    break
                tries, last = tries + 1, r
                if r.confident:
                    break
            if last is not None:
                out.append(dict(model=m, pipeline=P, view=v, photo_id=pid, group=last.group, tries=tries,
                                confident=bool(last.confident), correct=bool(last.correct)))
    return pd.DataFrame(out)

RUNS0 = runs_without_carry(IND)
isA = lambda d: d.group.astype(str).str.startswith("A")
isB = lambda d: d.group.astype(str).str.startswith("B")
pct = lambda s: float(s.mean() * 100) if len(s) else np.nan

# ---------- Table 1: each level on its own ----------
t1 = []
for m in MODELS_:
    for level in ORDER:
        g = IND[(IND.model == m) & (IND.level == level)]
        if not len(g):
            continue
        r = {"Model": m, "Level": level, "Tests": len(g), "Accuracy": pct(g.correct)}
        for v in ALL_VIEWS_USED:
            r[f"Accuracy: {v}"] = pct(g[g.view == v].correct) if (g.view == v).any() else np.nan
        r["Confident"] = pct(g.confident)
        r["Confident but wrong"] = pct(g.confident & ~g.correct)
        nv = g.view.nunique()
        full = g.groupby("photo_id").view.nunique() == nv
        same = g.groupby("photo_id").pred.nunique() == 1
        r["Same answer at every angle"] = pct(same[full]) if (nv > 1 and full.any()) else np.nan
        t1.append(r)
T1 = pd.DataFrame(t1).set_index(["Model", "Level"])

# ---------- Table 2: the three pipelines ----------
t2 = []
for m in MODELS_:
    for P in PIPES_:
        g = RUNS[(RUNS.model == m) & (RUNS.pipeline == P)] if len(RUNS) else pd.DataFrame()
        h = RUNS0[(RUNS0.model == m) & (RUNS0.pipeline == P)] if len(RUNS0) else pd.DataFrame()
        if len(g) and len(h):                 # compare on the same photos
            keys = set(zip(g.view, g.photo_id))
            h = h[[k in keys for k in zip(h.view, h.photo_id)]]
        if not len(g) and not len(h):
            continue
        n = np.nan
        st = g.stopped_at.value_counts(normalize=True) * 100 if len(g) else None
        r = {"Model": m, "Pipeline": P, "Starts at": PIPELINES[P][0], "Runs": len(g),
             "Average tries": g.tries.mean() if len(g) else n}
        for level in PIPELINES[P]:
            r[f"Stopped at {level} (%)"] = st.get(level, 0.0) if st is not None else n
        r["Never confident (%)"] = st.get("never", 0.0) if st is not None else n
        start = IND[(IND.model == m) & (IND.level == PIPELINES[P][0])]
        r["Start level alone: accuracy"] = pct(g.first_correct) if len(g) else pct(start.correct)
        r["Pipeline: final accuracy"] = pct(g.correct) if len(g) else n
        r["Nothing carried: runs"] = len(h)
        r["Nothing carried: average tries"] = h.tries.mean() if len(h) else n
        r["Nothing carried: final accuracy"] = pct(h.correct) if len(h) else n
        r["Confident but wrong (%)"] = pct(g.confident & ~g.correct) if len(g) else n
        r["Tries A"] = g[isA(g)].tries.mean() if len(g) else n
        r["Tries B"] = g[isB(g)].tries.mean() if len(g) else n
        r["Final accuracy A"] = pct(g[isA(g)].correct) if len(g) else n
        r["Final accuracy B"] = pct(g[isB(g)].correct) if len(g) else n
        t2.append(r)
T2 = pd.DataFrame(t2).set_index(["Model", "Pipeline"])

# ---------- Table 3: effect of carrying features (the bias check) ----------
t3 = []
if len(STEPS):
    S = STEPS.copy()
    S["m_order"] = S.model.map({m: i for i, m in enumerate(MODELS_)})
    S["p_order"] = S.pipeline.map({P: i for i, P in enumerate(PIPES_)})
    S = S.sort_values(["m_order", "p_order", "view", "photo_id", "step"])
    S["prev_pred"] = S.groupby(["model", "pipeline", "view", "photo_id"]).pred.shift(1)
    C = S[S.carried]
    if len(C):
        A = IND[["model", "view", "photo_id", "level", "pred", "correct"]]
        C = C.merge(A.rename(columns={"pred": "alone_pred", "correct": "alone_correct"}),
                    on=["model", "view", "photo_id", "level"], how="left")
        C = C.merge(A[["model", "view", "photo_id", "level", "pred"]].rename(columns={"level": "carried_from", "pred": "alone_prev_pred"}),
                    on=["model", "view", "photo_id", "carried_from"], how="left")
        C = C.sort_values(["m_order", "p_order", "step"], kind="stable")
        for (m, P, level), g in C.groupby(["model", "pipeline", "level"], sort=False):
            r = {"Model": m, "Pipeline": P, "Level": level, "From": g.carried_from.iloc[0], "Tests": len(g),
                 "With carried features: accuracy": pct(g.correct),
                 "Same photos alone: accuracy": pct(g.alone_correct.astype(float)),
                 "With carried features: confident": pct(g.confident)}
            r["Change (points)"] = r["With carried features: accuracy"] - r["Same photos alone: accuracy"]
            r["Kept previous answer (%)"] = pct(g.pred == g.prev_pred)
            r["Alone, same answer at both levels (%)"] = pct(g.alone_pred == g.alone_prev_pred)
            t3.append(r)
T3 = pd.DataFrame(t3).set_index(["Model", "Pipeline", "Level"]) if t3 else pd.DataFrame()

# ---------- Table 4: angles ----------
t4 = []
VIEWS_OF = {m: [v for v in ALL_VIEWS_USED if (IND[IND.model == m].view == v).any()] for m in MODELS_}
for m in MODELS_:
    for v in VIEWS_OF[m]:
        r = {"Model": m, "Angle": v}
        g = IND[(IND.model == m) & (IND.view == v)]
        for level in ORDER:
            r[f"{level} alone"] = pct(g[g.level == level].correct)
        for P in PIPES_:
            h = RUNS[(RUNS.model == m) & (RUNS.view == v) & (RUNS.pipeline == P)] if len(RUNS) else pd.DataFrame()
            r[f"{P} final"] = pct(h.correct) if len(h) else np.nan
        t4.append(r)
T4 = pd.DataFrame(t4).set_index(["Model", "Angle"])

# ---------- Table 5: info-rich (A) vs info-scarce (B) ----------
t5 = []
for m in MODELS_:
    for name, sel_ in [("A (info-rich)", isA), ("B (info-scarce)", isB)]:
        g = IND[(IND.model == m)]
        g = g[sel_(g)]
        r = {"Model": m, "Group": name}
        for level in ORDER:
            r[f"{level} alone"] = pct(g[g.level == level].correct)
        for P in PIPES_:
            h = RUNS[(RUNS.model == m) & (RUNS.pipeline == P)] if len(RUNS) else pd.DataFrame()
            h = h[sel_(h)] if len(h) else h
            r[f"{P} tries"] = h.tries.mean() if len(h) else np.nan
            r[f"{P} final"] = pct(h.correct) if len(h) else np.nan
        t5.append(r)
T5 = pd.DataFrame(t5).set_index(["Model", "Group"])

for title, t in [("1. Each level on its own (accuracy %, by angle)", T1), ("2. The three pipelines", T2),
                 ("3. Effect of carrying features from the previous level", T3), ("4. Angles (accuracy %)", T4),
                 ("5. Info-rich (A) vs info-scarce (B)", T5)]:
    print("\n" + title)
    display(t.round(1) if len(t) else "no data yet")

In [ ]:
# ---------- charts ----------
TAG = (GEM_MODEL if (RUN_GEMINI and GEM_MODEL) else "clip_only")   # output files are named after the Gemini model
INK, INK2, MUTED, GRIDC, AXISC = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
# one fixed colour per angle (the same angle always gets the same colour), colour-blind-safe order
ALL_VIEWS = ["original", "rot90", "rot180", "mirror", "rot270", "rot45", "tilt"]
VIEW_COLOUR = dict(zip(ALL_VIEWS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]))
UP, DOWN = "#2a78d6", "#e34948"       # better / worse
plt.rcParams.update({"font.family": "DejaVu Sans"})
FIGS = []

def tidy(ax):
    ax.grid(axis="y", color=GRIDC, lw=0.8); ax.set_axisbelow(True)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    for s in ["left", "bottom"]:
        ax.spines[s].set_color(AXISC)
    ax.tick_params(colors=INK2, labelsize=9)

def spread(ys, gap, top=101):
    # moves end labels apart so they don't overlap (and keeps them inside the chart); same order as ys
    order = np.argsort(ys)
    out = np.array(ys, float)
    for a, b in zip(order[:-1], order[1:]):
        out[b] = max(out[b], out[a] + gap)
    return out - max(0.0, out.max() - top)

# Chart 1: each level on its own, one line per angle
fig, axes = plt.subplots(1, len(MODELS_), figsize=(5.6 * len(MODELS_), 4.0), dpi=150, sharey=True)
axes = np.atleast_1d(axes)
x = np.arange(len(ORDER))
for ax, m in zip(axes, MODELS_):
    ends, mv = [], VIEWS_OF[m]
    for v in mv:
        y = [pct(IND[(IND.model == m) & (IND.view == v) & (IND.level == lv)].correct) for lv in ORDER]
        ax.plot(x, y, marker="o", ms=5, lw=1.8, color=VIEW_COLOUR.get(v, MUTED), label=v)
        ends.append(y[-1])
    if len(mv) <= 4 and not np.isnan(ends).any():       # names at the line ends
        for v, y0, yl in zip(mv, ends, spread(ends, 6.5)):
            ax.plot([x[-1] + 0.06, x[-1] + 0.2], [y0, yl], color=VIEW_COLOUR.get(v, MUTED), lw=0.8)
            ax.text(x[-1] + 0.24, yl, v, va="center", fontsize=8, color=INK2)
    ax.set_xticks(x, [f"{lv}\n(alone)" for lv in ORDER])
    ax.set_xlim(-0.25, len(ORDER) - 1 + 0.85)
    ax.set_ylim(0, 105); ax.set_yticks(range(0, 101, 20)); ax.set_title(m, loc="left", fontsize=11, color=INK)
    tidy(ax)
axes[0].set_ylabel("Accuracy (%)", color=INK2)
_most = max(range(len(MODELS_)), key=lambda i: len(VIEWS_OF[MODELS_[i]]))
fig.legend(*axes[_most].get_legend_handles_labels(), frameon=False, fontsize=8, loc="upper right",
           ncol=min(len(VIEWS_OF[MODELS_[_most]]), 4), bbox_to_anchor=(0.995, 0.995))
fig.suptitle("Each level on its own: accuracy at every angle", x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout()
FIGS.append(("chart1_levels_alone_by_angle.png", fig))

# Chart 2: pipelines vs their start level alone
if len(RUNS):
    fig, axes = plt.subplots(1, len(MODELS_), figsize=(6.2 * len(MODELS_), 4.2), dpi=150, sharey=True)
    axes = np.atleast_1d(axes)
    for ax, m in zip(axes, MODELS_):
        xs = np.arange(len(PIPES_))
        val = lambda P, c: float(T2.loc[(m, P), c]) if (m, P) in T2.index else np.nan
        a = [val(P, "Start level alone: accuracy") for P in PIPES_]
        c = [val(P, "Nothing carried: final accuracy") for P in PIPES_]
        c_tr = [val(P, "Nothing carried: average tries") for P in PIPES_]
        b = [val(P, "Pipeline: final accuracy") for P in PIPES_]
        b_tr = [val(P, "Average tries") for P in PIPES_]
        w = 0.27
        ax.bar(xs - w, a, w * 0.94, color=AXISC, label="start level alone")
        ax.bar(xs, c, w * 0.94, color="#86b6ef", label="pipeline, nothing carried")
        ax.bar(xs + w, b, w * 0.94, color=UP, label="pipeline, features carried")
        for xi, ai, ci, cti, bi, bti in zip(xs, a, c, c_tr, b, b_tr):
            if not np.isnan(ai):
                ax.text(xi - w, ai + 2, f"{ai:.0f}%", ha="center", fontsize=7.5, color=MUTED)
            if not np.isnan(ci):
                ax.text(xi, ci + 2, f"{ci:.0f}%\n{cti:.1f} tries", ha="center", fontsize=7.5, color=INK2)
            if not np.isnan(bi):
                ax.text(xi + w, bi + 2, f"{bi:.0f}%\n{bti:.1f} tries", ha="center", fontsize=7.5, color=INK2)
            else:
                ax.text(xi + w, 3, "not run\nyet", ha="center", fontsize=7, color=MUTED)
        ax.set_xticks(xs, [f"{P}\n(starts {PIPELINES[P][0]})" for P in PIPES_])
        ax.set_ylim(0, 120); ax.set_yticks(range(0, 101, 20)); ax.set_title(m, loc="left", fontsize=11, color=INK)
        tidy(ax)
    axes[0].set_ylabel("Accuracy (%)", color=INK2)
    fig.legend(*axes[0].get_legend_handles_labels(), frameon=False, fontsize=8, loc="upper right", ncol=3,
               bbox_to_anchor=(0.995, 0.995))
    fig.suptitle("The three pipelines: final accuracy and average tries", x=0.01, ha="left", fontsize=12, fontweight="bold")
    fig.tight_layout()
    FIGS.append(("chart2_pipelines.png", fig))

# Chart 3: effect of carrying features at each level
if len(T3):
    fig, axes = plt.subplots(1, len(MODELS_), figsize=(5.6 * len(MODELS_), 3.9), dpi=150, sharey=True)
    axes = np.atleast_1d(axes)
    allv = T3["Change (points)"].to_numpy(dtype=float)
    lo, hi = min(np.nanmin(allv), 0) if len(allv) else -1, max(np.nanmax(allv), 0) if len(allv) else 1
    pad = max(3.0, 0.2 * (hi - lo))
    for ax, m in zip(axes, MODELS_):
        if m not in T3.index.get_level_values(0):
            ax.axis("off")
            continue
        sub = T3.loc[m].reset_index()
        labels = [f"{r.Pipeline}\n{r.From}→{r.Level}" for r in sub.itertuples()]
        vals = sub["Change (points)"].to_numpy(dtype=float)
        ax.bar(np.arange(len(vals)), vals, 0.7, color=[UP if v >= 0 else DOWN for v in vals])
        for i, v in enumerate(vals):
            if not np.isnan(v):
                ax.text(i, v + (0.04 if v >= 0 else -0.04) * (hi - lo + 2 * pad), f"{v:+.0f}", ha="center",
                        va="bottom" if v >= 0 else "top", fontsize=8, color=INK2)
        ax.axhline(0, color=INK2, lw=0.8)
        ax.set_ylim(lo - pad, hi + pad)
        ax.set_xticks(np.arange(len(vals)), labels, fontsize=8)
        ax.set_title(m, loc="left", fontsize=11, color=INK)
        tidy(ax)
    axes[0].set_ylabel("Accuracy change (points)", color=INK2)
    fig.suptitle("Carrying the previous level's features: accuracy change vs the same photos alone",
                 x=0.01, ha="left", fontsize=12, fontweight="bold")
    fig.tight_layout()
    FIGS.append(("chart3_carry_effect.png", fig))

for name, fig in FIGS:
    name = f"{TAG}_{name}"
    fig.savefig(os.path.join(LOOP_DIR, name), facecolor="white", bbox_inches="tight")
    plt.close(fig)
    display(ShowImage(filename=os.path.join(LOOP_DIR, name), width=900))

In [ ]:
# ---------- "what to say": plain sentences with the numbers ----------
SAY = []
for m in MODELS_:
    lv = {level: pct(IND[(IND.model == m) & (IND.level == level)].correct) for level in ORDER}
    nvm = len(VIEWS_OF[m])
    SAY.append(f"{m}, each level on its own" + (f" (average over {nvm} angles)" if nvm > 1 else "") + ": " +
               ", ".join(f"{lv[l]:.0f}% at {l}" for l in ORDER) + ".")
    if nvm > 1:
        per_view = IND[IND.model == m].groupby("view").correct.mean() * 100
        same = T1.loc[(m, "L0"), "Same answer at every angle"] if (m, "L0") in T1.index else np.nan
        SAY.append(f"{m}, angles: accuracy over all levels ranged from {per_view.min():.0f}% ({per_view.idxmin()}) to "
                   f"{per_view.max():.0f}% ({per_view.idxmax()})"
                   + (f"; on clean photos it gave the same answer at every angle for {same:.0f}% of photos." if not np.isnan(same) else "."))
    if len(T2) and m in T2.index.get_level_values(0):
        parts = []
        for P in PIPES_:
            if (m, P) in T2.index:
                r = T2.loc[(m, P)]
                nc = (f"{r['Nothing carried: average tries']:.1f} tries and {r['Nothing carried: final accuracy']:.0f}% "
                      f"with nothing carried" if not np.isnan(r["Nothing carried: final accuracy"]) else "")
                if np.isnan(r["Average tries"]):
                    parts.append(f"{P} (starts at {PIPELINES[P][0]}): the steps with carried features haven't been asked "
                                 f"yet" + (f"; {nc}" if nc else ""))
                else:
                    parts.append(f"{P} (starts at {PIPELINES[P][0]}) needed {r['Average tries']:.1f} tries and was right "
                                 f"{r['Pipeline: final accuracy']:.0f}% of the time" + (f" ({nc})" if nc else "")
                                 + f", against {r['Start level alone: accuracy']:.0f}% for {PIPELINES[P][0]} alone")
        SAY.append(f"{m}, pipelines: " + "; ".join(parts) + ".")
    if len(T3) and m in T3.index.get_level_values(0):
        sub = T3.loc[m]
        w = np.average(sub["Change (points)"].fillna(0), weights=sub["Tests"])
        kept, kept0 = np.average(sub["Kept previous answer (%)"].fillna(0), weights=sub["Tests"]), \
                      np.average(sub["Alone, same answer at both levels (%)"].fillna(0), weights=sub["Tests"])
        SAY.append(f"{m}, carrying features: with the previous level's features, accuracy changed by {w:+.0f} points on "
                   f"average compared with the same photos judged alone. The model kept its previous answer {kept:.0f}% of "
                   f"the time, against {kept0:.0f}% when the two levels were judged separately"
                   + (" (a sign of anchoring bias)." if kept > kept0 + 5 else "."))
    if m in T5.index.get_level_values(0):
        a, b = T5.loc[(m, "A (info-rich)")], T5.loc[(m, "B (info-scarce)")]
        SAY.append(f"{m}, info-rich vs info-scarce: at L3 alone {a['L3 alone']:.0f}% vs {b['L3 alone']:.0f}%; "
                   f"at L0 alone {a['L0 alone']:.0f}% vs {b['L0 alone']:.0f}%"
                   + (f"; P3 needed {a['P3 tries']:.1f} vs {b['P3 tries']:.1f} tries." if "P3 tries" in T5.columns and not np.isnan(a["P3 tries"]) else "."))

display(Markdown("### What to say\n\n" + "\n\n".join("> " + s for s in SAY)))
with open(os.path.join(LOOP_DIR, f"{TAG}_what_to_say.txt"), "w", encoding="utf-8") as f:
    f.write("\n\n".join(SAY) + "\n")

# ---------- everything in one Excel file ----------
xlsx = os.path.join(LOOP_DIR, f"{TAG}_loop_v2_results.xlsx")
with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
    pd.DataFrame({"what to say": SAY}).to_excel(w, sheet_name="What to say", index=False)
    for name, t in [("1 levels alone", T1), ("2 pipelines", T2), ("3 carry effect", T3), ("4 angles", T4), ("5 groups", T5)]:
        if len(t):
            t.round(2).to_excel(w, sheet_name=name)
    IND.to_excel(w, sheet_name="all levels-alone tests", index=False)
    if len(STEPS):
        STEPS.to_excel(w, sheet_name="all pipeline steps", index=False)
        RUNS.to_excel(w, sheet_name="all pipeline runs", index=False)
print("Saved", xlsx)

## 8. Download the results

This makes `SafeCrop_loop_v2_results_<model>.zip` with all the tables, charts and answers, and downloads it. If Drive is connected, everything is also in `My Drive/SafeCrop/results/loop_v2`.

In [ ]:
zip_path = os.path.join(WORK, f"SafeCrop_loop_v2_results_{TAG}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in sorted(os.listdir(LOOP_DIR)):
        if not name.endswith(".npz"):            # CLIP's raw feature numbers stay in Drive (large, not needed)
            z.write(os.path.join(LOOP_DIR, name), name)
print("Zip ready:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Automatic download didn't start (" + short(e, 80) + "). Use the folder icon on the left to download it.")

## If something goes wrong

| What you see | What to do |
|---|---|
| A box asks *"Paste your Gemini API key"* | The secret is missing or Notebook access is off. Paste the key in the box and press Enter, or fix the secret (key icon on the left) and run again. |
| *"Gemini rejected the API key"* | The key was copied wrongly. Create a new one in AI Studio, update the `GEMINI_API_KEY` secret, then **Runtime → Restart session** and **Run all**. |
| *"gemini-3.5-flash-lite: not available"* | Look at the list of model names it prints, put the exact Flash-Lite name in `GEMINI_MODEL`, and **Run all**. |
| *"The free key's limit is used up"* | Normal: 500 a day. Run all again after the daily reset (about 1–2 PM Bangladesh time); nothing is asked twice. |
| *"too many requests"* | The notebook slows down by itself. If it stops, wait a minute and run section 6 again. |
| CLIP is very slow | Switch on a GPU (**Runtime → Change runtime type → T4 GPU**). CLIP's features are saved, so a re-run skips this step. |
| The run stopped halfway | Run section 6 again. Finished answers are kept and nothing is asked twice. |

**Files** (in `SafeCrop/results/loop_v2`):
* `gemini_v2_calls.csv` holds every Gemini answer, including its visible-signs notes. `kind = alone` marks a level judged on its own, and `kind = carry` marks a pipeline step with notes from the level before.
* `clip_levels_alone.csv` and `clip_pipeline_steps.csv` hold the CLIP results.
* `<model>_loop_v2_results.xlsx` has all the tables plus every individual test.
* The folder also has `<model>_what_to_say.txt` and the three charts.